<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TweetsGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from transformers import AutoModelForCausalLM,AutoModel,AutoTokenizer
import os
from google.colab import userdata
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

In [5]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [6]:
!kaggle datasets download -d suchintikasarkar/sentiment-analysis-for-mental-health

Dataset URL: https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health
License(s): DbCL-1.0
100% 11.1M/11.1M [00:00<00:00, 119MB/s]



In [7]:
!unzip -q sentiment-analysis-for-mental-health.zip -d ./datafolder/

In [10]:
df = pd.read_csv('/content/datafolder/Combined Data.csv')

In [11]:
df = df.dropna(subset=['statement'])

In [12]:
df = df[df['statement'].str.strip() != ""]

In [21]:
autoToken = AutoTokenizer.from_pretrained('gpt2')
# 2. THE CRITICAL FIX: GPT-2 needs a padding token
# We tell it to use the 'End of String' token as padding
autoToken.pad_token = autoToken.eos_token

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [14]:
max_len = 256
vocab_size = autoToken.vocab_size

In [16]:
statement_text = df['statement'].astype(str).values

In [17]:
x_train,x_test = train_test_split(statement_text,test_size=0.3,random_state=42)

In [22]:
train_data = autoToken(text=list(x_train),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_data = autoToken(text=list(x_test),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [26]:
class SentimentDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    ids = self.encoding['input_ids'][idx]
    mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':ids[:-1],
        'target_ids':ids[1:],
        'attention_mask':mask[:-1]
    }

In [27]:
train_d = SentimentDataset(encoding=train_data)
val_d = SentimentDataset(encoding=val_data)

In [28]:
train_ds = DataLoader(dataset=train_d,batch_size=64,shuffle=True,pin_memory=True,num_workers=2)
val_ds = DataLoader(dataset=val_d,batch_size=64,shuffle=False,pin_memory=True,num_workers=2)

GPT-PreTrained

In [29]:
class PreTrainedGPT(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    #Load the model
    self.gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')

  def forward(self,input_ids,attention_mask,labels=None):
    # GPT-2 calculates the loss internally if we pass labels
    outputs = self.gpt2(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
    return outputs.loss,outputs.logits

In [ ]:
model = PreTrainedGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
for epoch in range(3):
    model.train()
    for batch in train_ds:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['target_ids'].to(device)

        optimizer.zero_grad()
        # GPT2 calculates internal loss when you pass labels
        loss, logits = model(ids, mask, labels=labels)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        # 3. ADD Perplexity (The 'Accuracy' for writers)
        # At the end of the epoch, calculate this:
    avg_train_loss = train_loss / len(train_ds)
    train_perplexity = torch.exp(torch.tensor(avg_train_loss))

    print(f"Train Loss: {avg_train_loss:.4f} | Train Perplexity: {train_perplexity:.2f}")

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_ds:
            outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device), labels=batch['labels'].to(device))
            total_val_loss += outputs[0].item()

    avg_loss = total_val_loss / len(val_ds)
    print(f"Epoch {epoch} | Val Loss: {avg_loss:.4f} | Perplexity: {torch.exp(torch.tensor(avg_loss)):.2f}")

In [35]:
class CustomGPT2Writer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. The Pre-trained 'Writer' Brain (768 features)
        self.gpt2 = AutoModel.from_pretrained('gpt2')

        # 2. YOUR OWN CUSTOM LAYERS (Just like you wanted!)
        self.dropout = nn.Dropout(0.2)
        # For a writer, the output MUST be the vocab_size
        self.output_layer = nn.Linear(768, vocab_size)

    def forward(self, input_ids, attention_mask):
        # 1. Get features from the brain
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        # 2. Grab the hidden states (The words' meaning)
        # Shape: [Batch, Seq_Len, 768]
        x = outputs.last_hidden_state

        # 3. Apply your own custom logic
        x = self.dropout(x)

        # 4. Pass through your own custom output layer
        # Result: [Batch, Seq_Len, Vocab_Size]
        logits = self.output_layer(x)

        return logits

In [ ]:
# vocab_size is usually 50257 for GPT2
custommodel = CustomGPT2Writer(vocab_size=autoToken.vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
loss = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(3):
    model.train()
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device) # These are our "shifted" words

        optimizer.zero_grad()
        logits = model(ids, mask) # Your custom model only returns logits

        # CRITICAL: Flatten 3D [Batch, Seq, Vocab] -> 2D [Batch*Seq, Vocab] for CrossEntropy
        loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))

        loss.backward()
        optimizer.step()

    # Perplexity is calculated the same way here!